# Notebook 09 — Louvain topology arm

**Goal:** Run the topology-aware aggregation arm on the Louvain community split — the one place
where FedAvg already underperforms local-only (FedAvg 0.7308 < local 0.7423 at k=5,
from `docs/2026-08-11-splits-and-results.md` §3/§7). The topology arm has never been run here.

**Design (pre-specified before running cells 6–9):**
- Split: `louvain_split`, k=5 (reproduce known FedAvg-fails result) and k=10 (larger split per FedGTA advice).
- Arms: FedAvg vs topology-aware (`distinctiveness`, T=1.0) — same mode used in Phase 5 and Experiment #1.
- Seeds: 42–46 (5), matched budget: `rounds=8, local_epochs=4, local_only_epochs=32, hidden=128`.
- Criteria (same shape as pre-registered criteria from nb 08):
  - **C1:** `fedavg_worst ≤ local_worst − 0.01` — FedAvg visibly hurts the minority (condition reproduces)
  - **C2:** `topo − fedavg ≥ +0.005` on worst-hospital OR mean — topology rescues the gap
  - **C3:** tail-hospital F1 under topo > under FedAvg — rescue comes from raising the tail, not sinking others

**Do NOT re-tune the split or criteria after seeing results.** Any iteration = appended cell with date + rationale.

---

In [ ]:
# 1) Get the code + deps (record versions — deps are unpinned)
!git clone -b main https://github.com/RawEgg6/Capstone-amr-fed.git 2>/dev/null || (cd Capstone-amr-fed && git fetch && git checkout main && git pull)
!cd /content/Capstone-amr-fed && git log -1 --oneline
!pip install -q torch_geometric 'flwr[simulation]' networkx
!pip freeze | grep -E '^(torch|torch-geometric|flwr|ray|networkx)==' || true

In [ ]:
# 2) Mount Drive + set ARMD_DIR
from google.colab import drive
drive.mount('/content/drive')
import os
os.environ['ARMD_DIR'] = '/content/drive/MyDrive/ARMD'   # edit to your folder if different

In [ ]:
# 3) Sanity: data resolves + imports
import sys
sys.path.insert(0, '/content/Capstone-amr-fed/src')
from amr_fed import config
from pathlib import Path
D = Path(config.DATA_DIR)
print('DATA_DIR:', D, '| exists:', D.exists())
assert D.exists(), 'ARMD_DIR is wrong — fix cell 2 and re-run.'
from amr_fed.federated.run import run_multiseed
from amr_fed.data_loader import load_cohort_frame, PK
from amr_fed.partition import louvain_split, partition_summary
print('imports OK')

In [ ]:
# 4) Load cohort ONCE — reused by every arm cell below
import sys as _s
for _m in [m for m in list(_s.modules) if m.startswith('amr_fed')]:
    del _s.modules[_m]
from amr_fed.data_loader import load_cohort_frame
df = load_cohort_frame()
print('rows:', f'{len(df):,}')

In [ ]:
# 5) SMOKE: validate both Louvain splits before any training
import sys as _s
for _m in [m for m in list(_s.modules) if m.startswith('amr_fed')]:
    del _s.modules[_m]
from amr_fed.data_loader import load_cohort_frame, PK
from amr_fed.partition import louvain_split, partition_summary

for k in (5, 10):
    print(f'\n--- Louvain k={k} ---')
    assign = louvain_split(df, n_clients=k)
    actual_k = assign.nunique()
    g, niid = partition_summary(df, assign)
    print(g)
    print(f'non-IID(wstd): {niid:.4f} | actual hospitals: {actual_k}')
    print(f'resist_rate spread: {g["resist_rate"].max() - g["resist_rate"].min():.4f}')
    assert assign.index.is_unique and len(assign) == df[PK].nunique(), 'not 1:1'
    assert (g['n_tests'] / g['n_tests'].sum() >= 0.01).all(), 'hospital below 1% size floor'
    print(f'SMOKE PASS k={k}')

print('\nBoth splits valid — safe to run arms.')

In [ ]:
# 6) LOUVAIN k=5, FedAvg arm (5 seeds)
# Expected baseline: FedAvg mean ~0.73, local ~0.74 — FedAvg should LOSE here.
import os, ray
os.environ.pop('AMR_RNG_PROBE', None)   # must NOT be set during arms
os.environ['AMR_DEBUG_INIT'] = '1'
ray.shutdown()
import sys as _s
for _m in [m for m in list(_s.modules) if m.startswith('amr_fed')]:
    del _s.modules[_m]
from amr_fed.federated.run import run_multiseed
from amr_fed.partition import louvain_split

lv5_fed = run_multiseed(
    seeds=(42, 43, 44, 45, 46), df=df,
    label='louvain-k5/FedAvg',
    partition_fn=lambda d, s: louvain_split(d, n_clients=5, seed=s),
    n_clients=5, rounds=8, local_epochs=4,
    local_only_epochs=32, hidden=128
)

In [ ]:
# 7) LOUVAIN k=5, topology arm (THE KEY RUN — never done before on Louvain)
import os, ray
os.environ.pop('AMR_RNG_PROBE', None)
os.environ['AMR_DEBUG_INIT'] = '1'
ray.shutdown()
import sys as _s
for _m in [m for m in list(_s.modules) if m.startswith('amr_fed')]:
    del _s.modules[_m]
from amr_fed.federated.run import run_multiseed
from amr_fed.partition import louvain_split

lv5_topo = run_multiseed(
    seeds=(42, 43, 44, 45, 46), df=df,
    label='louvain-k5/TOPO',
    partition_fn=lambda d, s: louvain_split(d, n_clients=5, seed=s),
    n_clients=5, rounds=8, local_epochs=4,
    local_only_epochs=32, hidden=128,
    strategy='topology', topo_mode='distinctiveness', topo_temperature=1.0
)

In [ ]:
# 8) LOUVAIN k=10, FedAvg arm
# Larger split -> smaller, more structurally isolated communities -> FedAvg penalty may grow.
import os, ray
os.environ.pop('AMR_RNG_PROBE', None)
os.environ['AMR_DEBUG_INIT'] = '1'
ray.shutdown()
import sys as _s
for _m in [m for m in list(_s.modules) if m.startswith('amr_fed')]:
    del _s.modules[_m]
from amr_fed.federated.run import run_multiseed
from amr_fed.partition import louvain_split

lv10_fed = run_multiseed(
    seeds=(42, 43, 44, 45, 46), df=df,
    label='louvain-k10/FedAvg',
    partition_fn=lambda d, s: louvain_split(d, n_clients=10, seed=s),
    n_clients=10, rounds=8, local_epochs=4,
    local_only_epochs=32, hidden=128
)

In [ ]:
# 9) LOUVAIN k=10, topology arm
import os, ray
os.environ.pop('AMR_RNG_PROBE', None)
os.environ['AMR_DEBUG_INIT'] = '1'
ray.shutdown()
import sys as _s
for _m in [m for m in list(_s.modules) if m.startswith('amr_fed')]:
    del _s.modules[_m]
from amr_fed.federated.run import run_multiseed
from amr_fed.partition import louvain_split

lv10_topo = run_multiseed(
    seeds=(42, 43, 44, 45, 46), df=df,
    label='louvain-k10/TOPO',
    partition_fn=lambda d, s: louvain_split(d, n_clients=10, seed=s),
    n_clients=10, rounds=8, local_epochs=4,
    local_only_epochs=32, hidden=128,
    strategy='topology', topo_mode='distinctiveness', topo_temperature=1.0
)

In [ ]:
# 10) VERDICT — C1/C2/C3 for both k=5 and k=10
import json, numpy as np

def _tail_f1(res, key):
    """Per-seed F1 of the tail hospital (highest client id)."""
    vals = []
    for r in res.get('runs', []):
        pc = r.get(key)
        if not pc:
            continue
        vals.append(pc[max(pc, key=lambda x: int(x))] if isinstance(pc, dict) else pc[-1])
    return vals

results = {}
for label, fed, topo in [
    ('k=5',  lv5_fed,  lv5_topo),
    ('k=10', lv10_fed, lv10_topo),
]:
    print(f'\n====== Louvain {label} ======')
    d_mean  = round((topo['fedavg_best'][0]  or 0) - (fed['fedavg_best'][0]  or 0), 4)
    d_worst = round((topo['worst_fed'][0]    or 0) - (fed['worst_fed'][0]    or 0), 4)
    c1 = (fed['worst_fed'][0] or 0) <= (fed['worst_local'][0] or 0) - 0.01
    c2 = (d_worst >= 0.005) or (d_mean >= 0.005)

    tail_f = _tail_f1(fed, 'fedavg_per_client')
    tail_t = _tail_f1(topo, 'fedavg_per_client')
    tail_l = _tail_f1(fed, 'local_only_per_client')
    tf = round(float(np.mean(tail_f)), 4) if tail_f else None
    tt = round(float(np.mean(tail_t)), 4) if tail_t else None
    tl = round(float(np.mean(tail_l)), 4) if tail_l else None
    c3 = (tt is not None and tf is not None and tt > tf)

    print(f"fedavg   best {fed['fedavg_best']} | worst {fed['worst_fed']} | worst_local {fed['worst_local']}")
    print(f"topo     best {topo['fedavg_best']} | worst {topo['worst_fed']}")
    print(f"C1 FedAvg hurts minority (worst_fed <= worst_local - 0.01): {'PASS' if c1 else 'FAIL'}")
    print(f"C2 Path-B gate (delta_mean {d_mean} | delta_worst {d_worst}): {'PASS' if c2 else 'FAIL'}")
    print(f"C3 tail hospital F1: local {tl} | fedavg {tf} | topo {tt} -> {'PASS' if c3 else 'FAIL'}")

    payload = {
        'C1': bool(c1), 'C2': bool(c2), 'C3': bool(c3),
        'delta_mean': d_mean, 'delta_worst': d_worst,
        'tail_local': tl, 'tail_fedavg': tf, 'tail_topo': tt,
        'tail_fedavg_per_seed': tail_f, 'tail_topo_per_seed': tail_t,
        'fed':  {'best': fed['fedavg_best'],  'worst': fed['worst_fed'],
                 'worst_local': fed['worst_local'], 'final': fed.get('fedavg_final')},
        'topo': {'best': topo['fedavg_best'], 'worst': topo['worst_fed'],
                 'worst_local': topo['worst_local'], 'final': topo.get('fedavg_final')},
    }
    results[label] = payload
    path = f'/content/drive/MyDrive/amr_louvain_{label.replace("=","")}_verdict.json'
    with open(path, 'w') as f:
        json.dump(payload, f, default=str, indent=1)
    print('saved', path)

print('\n=== COPY EVERYTHING BELOW THIS LINE BACK TO CHAT ===')
print(json.dumps(results, default=str))
print('=== END ===')